# 🚀 MEM LLM Orchestrator — Live Interactive Demo
### Autonomous Adaptive GPU Orchestration, Dynamic Lane Switching & Zero-OOM Defense

Welcome to the interactive testbench for **MEM Orchestrator**!
This notebook lets you experience the autonomous memory governor and adaptive lane switching engine in action on Google Colab cloud GPUs (L4 / T4 / A100) or CPU.

**What you will see in this demo:**
1. **Dynamic Hardware Calibration:** Auto-detects available GPU VRAM and tunes throughput lanes.
2. **Adaptive Lane Switching:** Watches the model promote/demote batch size and gradient accumulation in real-time.
3. **Zero-OOM Chaos Resilience:** Injects synthetic VRAM spikes (+10 GB shocks) and watches the MEM Governor prevent out-of-memory crashes on the fly.
4. **Autoregressive Text Generation:** Generates text using the trained weights.

In [ ]:
#@title 1. Setup & Hardware Discovery
import os, sys, torch

print("=" * 65)
print("  MEM ORCHESTRATOR - HARDWARE DISCOVERY")
print("=" * 65)
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"  GPU Detected: {device_name} ({vram_gb:.2f} GB VRAM)")
else:
    print("  Running on CPU (GPU acceleration not active)")
print("=" * 65)

# Clone repository if running in Colab
!git clone https://github.com/nobazzy/mem-llm-orchestrator.git 2>/dev/null || (cd mem-llm-orchestrator && git pull)
%cd /content/mem-llm-orchestrator/mem_v3

!pip install -q transformers datasets accelerate truststore
print("\n>>> Environment ready! Proceed to step 2 or 3.")

### 2. Run Live Adaptive Training
Launch the adaptive training loop. Watch how throughput, loss, and memory allocation stabilize in real time.

In [ ]:
#@title Launch Adaptive Training
import os
if os.path.exists("/content/mem-llm-orchestrator/mem_v3"):
    %cd /content/mem-llm-orchestrator/mem_v3
elif os.path.exists("mem_v3"):
    %cd mem_v3

steps = 200 #@param {type:"slider", min:50, max:1000, step:50}
model_preset = "medium_75m" #@param ["medium_75m", "large_130m", "xlarge_250m"]
dataset = "roneneldan/TinyStories" #@param ["roneneldan/TinyStories", "HuggingFaceFW/fineweb-edu"]

!python -u scripts/demo_video_chaos_defense.py --steps {steps} --model-preset {model_preset} --dataset {dataset} --no-shock

### 3. 💥 Stress Test: Chaos Shock Injection (Zero-OOM Defense)
Now we inject simulated memory shocks (+10,000MB / 10GB VRAM) during live training.
Watch how the **AdaptiveLaneRunner** instantly detects memory pressure, performs an **emergency demotion**, defragments VRAM, and **keeps the training alive** without throwing a `CUDA Out of Memory` exception!

In [ ]:
#@title Chaos Defense Demo (Simulated Hardware Shocks)
import os
if os.path.exists("/content/mem-llm-orchestrator/mem_v3"):
    %cd /content/mem-llm-orchestrator/mem_v3
elif os.path.exists("mem_v3"):
    %cd mem_v3

chaos_steps = 500 #@param {type:"slider", min:50, max:1000, step:50}
shock_size_mb = 10000 #@param {type:"slider", min:500, max:15000, step:500}

!python -u scripts/demo_video_chaos_defense.py --steps {chaos_steps} --model-preset medium_75m --dataset roneneldan/TinyStories --shock-interval 40 --shock-duration 20 --shock-size-mb {shock_size_mb}

### 4. 🧠 Text Generation & Inference Test
Test text generation autoregressively using the trained model weights and custom prompts:

In [ ]:
#@title Generate Text
import os
if os.path.exists("/content/mem-llm-orchestrator/mem_v3"):
    %cd /content/mem-llm-orchestrator/mem_v3
elif os.path.exists("mem_v3"):
    %cd mem_v3

prompt = "Once upon a time in a futuristic city," #@param {type:"string"}
temperature = 0.7 #@param {type:"slider", min:0.1, max:1.5, step:0.1}
max_tokens = 80 #@param {type:"integer"}

!python -u scripts/run_inference.py --prompt "{prompt}" --temperature {temperature} --max-tokens {max_tokens}